# Arquitectura de la CNN (consulta al profesor)

CNN propia desde cero (`src/models/cnn.py`), sin pesos preentrenados. Entrada: **un corte** con sus tres fases de contraste como canales (no es RGB). La salida es un logit por corte; la probabilidad de la paciente es la **media de sus cortes**, calculada fuera de la red.

```
Entrada (3, 256, 256)  PRE, EARLY, LATE   (valores en [0, 1])
  │
  ├─ Preprocesado (sin pesos): opcionalmente normalizar y añadir EARLY−PRE, LATE−EARLY
  │
  ├─ Bloque 1:  [Conv(→16) BN ReLU] [Conv(16→16) BN ReLU] MaxPool  →  (16, 128, 128)
  ├─ Bloque 2:  [Conv(→32) BN ReLU] [Conv(32→32) BN ReLU] MaxPool  →  (32, 64, 64)
  ├─ Bloque 3:  [Conv(→64) BN ReLU] [Conv(64→64) BN ReLU] MaxPool  →  (64, 32, 32)
  ├─ Bloque 4:  [Conv(→128) BN ReLU] [Conv(128→128) BN ReLU] MaxPool → (128, 16, 16)
  │
  ├─ Resumen del mapa: media (por defecto), máximo, media+máximo o atención  →  (128,)
  ├─ Dropout 0,5
  └─ Lineal 128 → 1  →  logit  (la sigmoide se aplica fuera: probabilidad de pCR)
```

| Pieza | Detalle | Por qué |
|---|---|---|
| Convoluciones | 3×3, padding 1, sin bias (lo pone BatchNorm) | Bloque clásico tipo VGG pequeño, escrito a mano |
| Preprocesado dentro de la red | `--realce` añade EARLY−PRE (captación) y LATE−EARLY (lavado); `--normalizar` = z-score por corte con **una** media y desviación para las tres fases; `--sin-late` quita LATE | La app de la defensa aplica exactamente el mismo preprocesado. Normalizar cada fase por separado borraría la captación |
| Resumen del mapa | `media`, `max`, `mediamax`, `atencion` (conv 1×1 + softmax sobre 16×16; empieza igual que la media) | El tumor ocupa poca superficie y la media lo diluye |
| Tamaño | **294.129 parámetros** (red base) | |
| Entrenamiento | AdamW, lr 1e-3, wd 1e-4, lote 32, semilla 42, hasta 40 épocas, parada con paciencia 8 sobre el AUC de validación por paciente | |
| Pérdida | BCE; `--ponderada` usa pos_weight = 2,40 (pCR = 1 es el 29,4 %) | |
| Aumentado | `--aumentado`: volteos y rotaciones de hasta 15°, iguales para los tres canales | |
| Partición | Por paciente con la columna `fold` de `samples.csv`; hasta ahora solo **fold 0** (validación: 219 pacientes) | |

## Comprobaciones hechas y lo que dicen

Todas en el fold 0, con AUC **por paciente** en validación. *Mejor AUC* = la de la mejor época; *AUC medio* = media de todas las épocas. Referencia de pérdida: predecir siempre 0,5 con la pérdida ponderada da **0,979**.

### 1. pCR con la red base y variantes

| Experimento | Épocas | Mejor AUC (época) | AUC medio | Pérdida val mínima | Pérdida train (1.ª → última) |
|---|---|---|---|---|---|
| Ponderada | 9 | 0,58 (1) | 0,54 | 0,980 | 0,98 → 0,87 |
| + aumentado | 9 | 0,58 (1) | 0,53 | 0,978 | 0,98 → 0,93 |
| + aumentado, lr 3e-4 | 9 | 0,58 (1) | 0,52 | 0,992 | 0,98 → 0,91 |
| + aumentado, realce | 26 | 0,59 (18) | 0,53 | 0,981 | 0,98 → 0,83 |
| + aumentado, normalizar | 25 | 0,55 (17) | 0,51 | 0,975 | 0,98 → 0,86 |
| + aumentado, pooling max | 9 | 0,51 (1) | 0,48 | 0,975 | 1,12 → 0,95 |
| + aumentado, atención | 14 | 0,58 (6) | 0,53 | 0,975 | 0,98 → 0,92 |

- La pérdida de validación **nunca baja claramente de la de un modelo constante** (0,979) mientras la de train sigue bajando: memoriza en lugar de aprender.
- El AUC salta de una época a otra (0,47-0,59).
- **Simulación de control:** un modelo *sin ninguna señal* da 0,56-0,58 si nos quedamos con la mejor de 9-25 épocas (±0,04 por época con 219 pacientes). Las diferencias entre variantes son ruido.
- Con realce o normalizar la mejor época llega tarde (17-18): el aumentado frena la memorización, pero no aparece señal.
- AUC por cohorte parecido (0,55-0,59); spy1 solo tiene 21 pacientes (4 pCR) en validación, no es interpretable.

### 2. ¿Funciona el circuito? Objetivo fácil: cohorte Duke

Misma red y mismo código con el objetivo «¿es Duke?»: **AUC 0,97 en la época 1 y 0,998 en la 13** (41/42 Duke acertadas, 0 falsos positivos).
- El código, la carga de datos y la agregación por paciente funcionan: el ~0,5 en pCR no es un error.
- Aviso: la red reconoce el hospital trivialmente, así que en pCR puede usar la cohorte como atajo.
- Con lr 1e-3 hay épocas con picos de pérdida de validación (1,98) y la sensibilidad a 0,5 se hunde aunque el AUC se mantiene: las probabilidades se desplazan en bloque entre épocas.

### 3. ¿Ve el tumor? Objetivo intermedio: subtipo tumoral

El subtipo debería ser más fácil que la pCR (deja huella en la imagen y está muy ligado a ella).

| Objetivo | Mejor AUC | AUC medio | Pérdida val mínima | Pérdida de un modelo constante |
|---|---|---|---|---|
| Triple negativo | 0,57 | 0,53 | 0,668 | 0,656 |
| HER2+ | 0,57 | 0,53 | 0,540 | 0,534 |
| HR+ | 0,57 | 0,52 | 0,714 | 0,677 |

Tampoco lo aprende: 0,57 es lo que da el azar al elegir la mejor época, y dentro de spy2 sola queda en 0,54-0,57. **Conclusión: con la mama entera a 256 px la red ve el hospital pero no el tumor.**

### 4. Localización del tumor (`notebooks/localizacion_tumor.ipynb`)

- Las columnas `sraw/eraw/scol/ecol` **sí** marcan el tumor, pero giradas 90° y en la escala del volumen original. Corregidas, la caja es fiable con n_xy ≤ 320 (~89 % de las pacientes, revisado a ojo en ~40); con n_xy 384-512 (casi todas de spy2) no cae en el tumor.
- La caja **no llega a la app** (en la defensa solo hay imágenes): sirve para entrenar o diagnosticar, pero el modelo final tiene que localizar el tumor solo.

### 5. Referencias de comparación (EDA)

- Variables clínicas (subtipo + edad + volumen), regresión logística: **AUC 0,69**. Solo la cohorte: 0,55. Medidas globales de realce: ~0,5.
- Cortes de una misma paciente muy correlacionados (0,81).

### Lo que aún falta (lo tenemos identificado)

- Solo fold 0; faltan los 5 folds con media ± desviación.
- No se registra el AUC de **train** por época, solo el de validación.
- En pCR no hay corrida con pérdida **normal** para compararla con la ponderada.
- Test sin tocar.

## Preguntas

1. **Mama entera o tumor.** Con la mama entera a 256 px la red no aprende ni el subtipo. ¿Es razonable centrarse en el tumor? Como la caja no llega a la app, había pensado usarla solo al entrenar (por ejemplo, para supervisar la atención o como pérdida auxiliar de localización) y que en la app la red localice el tumor sola. ¿Le parece bien, o hay una forma más sencilla?
2. **Qué cortes usar.** Ahora todos los cortes de una paciente heredan su etiqueta aunque no contengan tumor (etiquetas ruidosas). ¿Conviene quedarse solo con los cortes con tumor, o meter varios cortes vecinos como canales (2,5D)?
3. **Tamaño de la red.** 294.129 parámetros, 4 bloques, mapa final de 16×16. ¿Le parece adecuada, demasiado grande para unas 1.000 pacientes de entrenamiento, o le falta campo receptivo?
4. **Resumen del mapa.** ¿Media, máximo o atención? En nuestras pruebas ninguno cambia el resultado.
5. **Atajo de la cohorte.** La red distingue Duke con AUC 0,998. ¿Hay que hacer algo para que no lo use (normalizar por cohorte, evaluar por cohorte...) o basta con discutirlo en el informe?
6. **Siguiente paso.** Antes de lanzar los 5 folds, ¿qué cambiaría primero para acercarnos a su curva «BIEN»?